In [1]:
import itertools
import json
import math
import re
import sys
from collections import Counter

# Набор констант

# Сокращения, после которых точка не означает конец предложения
ABBR = r"(?:т\.\s?е|т\.\s?д|т\.\s?п|т\.\s?к|ул|гг|вв|др|см|рис|стр|тыс|млн|млрд|руб|[А-ЯЁA-Z])"

# Приеобразовываем текст в массив предложений
def split_sentences(text):
    # Нормализуем пробелы
    text = re.sub(r"\s+", " ", text.strip())

    # Теперь пока заменим все точки в сокращениях на специальный символ

    protected = re.sub(
        r"\b(" + ABBR + r")\.",
        lambda m: m.group(1) + "\u0000",
        text
    )

    # Также замечняем точки на специальный символ  после сокращения в конце дат

    protected = re.sub(
        r"(?<=\d)\s(г|гг)\.",
        lambda m: " " + m.group(1) + "\u0000",
        protected
    )

    # Теперь вводим разделения по символам конца предложений, заглавные буквы, числа

    parts = re.split(
        r"(?<=[.!?…])\s+(?=[«\"“(\[]?[А-ЯЁA-Z0-9])",
        protected
    )

    # Вводим возврат точек

    result = []
    for p in parts:
        p = p.replace("\u0000", ".").strip()
        if p:  # убираем пустые строки
            result.append(p)

    return result

In [2]:
# ТЕСТЫ

tests = [
    # Тест 1: Простой текст
    "Сегодня в Москве прошло заседание. Министр выступил с докладом.",

    # Тест 2: Сокращения
    "Правительство утвердило бюджет, т.е. все расходы были одобрены. Это важно, т.к. влияет на экономику.",

    # Тест 3: Инициалы
    "И.И. Иванов и А.С. Петров встретились в Москве.",

    # Тест 4: Даты и единицы
    "В 1990 г. родился известный ученый. Он получил 5 млн. руб. премии.",

    # Тест 5: Разные знаки препинания
    "Кто это сделал? Никто не знал! Но все догадались…",

    # Тест 6: Реальная новость из датасета
    "Министерство финансов РФ представило проект бюджета на 2024 г. "
    "Основные расходы, т.е. около 70% всех трат, пойдут на социальные нужды. "
    "Министр А.Г. Силуанов отметил, что это важно, т.к. нужно поддержать граждан."
]

for i, text in enumerate(tests, 1):
    print(f"\n{'='*60}")
    print(f"ТЕСТ {i}")
    print(f"{'='*60}")
    print(f"ВХОД:\n{text}\n")
    sents = split_sentences(text)
    print(f"ВЫХОД ({len(sents)} предложений):")
    for j, s in enumerate(sents, 1):
        print(f"  {j}. {s}")


ТЕСТ 1
ВХОД:
Сегодня в Москве прошло заседание. Министр выступил с докладом.

ВЫХОД (2 предложений):
  1. Сегодня в Москве прошло заседание.
  2. Министр выступил с докладом.

ТЕСТ 2
ВХОД:
Правительство утвердило бюджет, т.е. все расходы были одобрены. Это важно, т.к. влияет на экономику.

ВЫХОД (2 предложений):
  1. Правительство утвердило бюджет, т.е. все расходы были одобрены.
  2. Это важно, т.к. влияет на экономику.

ТЕСТ 3
ВХОД:
И.И. Иванов и А.С. Петров встретились в Москве.

ВЫХОД (1 предложений):
  1. И.И. Иванов и А.С. Петров встретились в Москве.

ТЕСТ 4
ВХОД:
В 1990 г. родился известный ученый. Он получил 5 млн. руб. премии.

ВЫХОД (2 предложений):
  1. В 1990 г. родился известный ученый.
  2. Он получил 5 млн. руб. премии.

ТЕСТ 5
ВХОД:
Кто это сделал? Никто не знал! Но все догадались…

ВЫХОД (3 предложений):
  1. Кто это сделал?
  2. Никто не знал!
  3. Но все догадались…

ТЕСТ 6
ВХОД:
Министерство финансов РФ представило проект бюджета на 2024 г. Основные расходы, т.е. 